# Data Visualization — Quiz (Lectures 1–7)

**Time:** 90 minutes
**Tools:** Plotly only (`plotly.express` first, then `update_traces` / `update_layout` / `update_xaxes` / `update_yaxes`)

**Your task:** answer each analytical question with **one polished, explanatory visualization**. For every question **you decide which chart type fits best** — and justify the choice in one or two sentences.

**Every figure must have:**
1. An **insight title** that states the finding, not just the variables
2. `template='plotly_white'` and a decluttered layout (no unnecessary gridlines, legends or axis titles)
3. A **CVD-safe palette** — no red/green; grey for context + one accent colour where it helps
4. Colour that **encodes meaning**, not decoration
5. Direct labels or annotations where they help the reader

**Chart types covered in the course** — choose from these:
line chart · sorted bar chart · stacked / 100% stacked bar chart · scatter plot · bubble chart · histogram · box plot · violin plot · choropleth map · treemap

> 💡 Each question has one chart type that fits it best, and the questions together cover most of the list above. If two of your answers use the same chart type, check the question again.


**Submit:** this notebook pushed to your personal repo as `quiz/dataviz_quiz.ipynb`

## The dataset — World Population 1970–2022

**Source:** [World Population Dataset](https://www.kaggle.com/datasets/iamsouravbanerjee/world-population-dataset) on Kaggle, compiled from Worldometer (UN World Population Prospects estimates). **234 rows × 16 columns — one row per country or territory, no missing values.**

| Column | Type | Description |
|---|---|---|
| `Country`, `Capital` | categorical | Country / territory and its capital |
| `Continent` | categorical | Africa, Asia, Europe, North America, Oceania, South America |
| `CCA3` | spatial | ISO-3 country code — use with `locationmode='ISO-3'` |
| `1970 Population` … `2022 Population` | temporal | Population in 1970, 1980, 1990, 2000, 2010, 2015, 2020, 2022 |
| `Area` | numerical | Land area in km² |
| `Density` | numerical | People per km² (2022) |
| `World share (%)` | numerical | Share of world population in 2022 |
| `Rank` | numerical | Rank by 2022 population |

**The next setup cell also gives you `df_long` — the same data in **tidy (long) format** with one row per country per year. Use `df` for one-year questions and `df_long` for questions about change over time**

In [12]:
import pandas as pd
import plotly.express as px
import numpy as np

df = pd.read_csv('world_population.csv')

# Shorter column names
df = df.rename(columns={'Country/Territory': 'Country', 'Area (km²)': 'Area',
                        'Density (per km²)': 'Density', 'World Population Percentage': 'World share (%)'})

# Tidy (long) version: one row per country per year
year_cols = [c for c in df.columns if c.endswith('Population')]
df_long = df.melt(id_vars=['CCA3', 'Country', 'Continent'], value_vars=year_cols,
                  var_name='Year', value_name='Population')

df_long['Year'] = df_long['Year'].str[:4].astype(int)     # '2022 Population' → 2022

print(df.shape, df_long.shape)


(234, 16) (1872, 5)


---
## Q1 — Which continent is growing fastest?

How has the population of each continent **grown since 1970, relative to its 1970 size** — and which continent is growing fastest?


**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

A line chart is suitable because it shows how population growth changes over time. Using a growth index (1970 = 100) allows us to compare the relative population growth of continents with different starting populations.

In [13]:

continent_year = (
    df_long.groupby(['Continent', 'Year'])['Population']
    .sum()
    .reset_index()
)


base_1970 = (
    continent_year[continent_year['Year'] == 1970]
    .set_index('Continent')['Population']
)


continent_year['Growth Index'] = (
    continent_year['Population'] /
    continent_year['Continent'].map(base_1970) * 100
)


latest_year = continent_year['Year'].max()

ranking = (
    continent_year[continent_year['Year'] == latest_year]
    .sort_values('Growth Index', ascending=False)
)

print(ranking[['Continent', 'Growth Index']])


fig = px.line(
    continent_year,
    x='Year',
    y='Growth Index',
    color='Continent',
    markers=True,
    title=f"{ranking.iloc[0]['Continent']} has the highest relative population growth",
    template='plotly_white'
)

fig.update_layout(
    yaxis_title='Population growth index (1970 = 100)',
    xaxis_title=None,
    legend_title=None
)

fig.show()

        Continent  Growth Index
7          Africa    390.409905
39        Oceania    231.200871
47  South America    226.391835
15           Asia    220.120725
31  North America    190.307634
23         Europe    113.297813


---
## Q2 — Which countries added the most people?

Which **15 countries added the most people** between 1970 and 2022 — and how many of them are in **Africa**?

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

A sorted horizontal bar chart is appropriate because it clearly compares the 15 countries with the largest population increases. Using different colours for continents helps identify the geographical distribution of these countries.


In [14]:
# Q2 - Countries adding the most people

df['Population Added'] = (
    df['2022 Population'] - df['1970 Population']
)

top15 = df.nlargest(15, 'Population Added').sort_values(
    'Population Added'
)

top15['Region'] = top15['Continent'].apply(
    lambda x: 'Africa' if x == 'Africa' else 'Other continents'
)

africa_count = (top15['Continent'] == 'Africa').sum()

print("Number of African countries in the top 15:", africa_count)
print(top15[['Country', 'Population Added', 'Continent']])

fig = px.bar(
    top15,
    x='Population Added',
    y='Country',
    orientation='h',
    color='Region',
    color_discrete_map={
        'Africa': '#0072B2',
        'Other continents': '#999999'
    },
    text='Population Added',
    title='Four African countries are among the 15 largest population gainers',
    template='plotly_white'
)

fig.update_traces(texttemplate='%{text:.3s}', textposition='outside')
fig.update_layout(
    xaxis_title=None,
    yaxis_title=None,
    legend_title=None
)
fig.update_xaxes(showgrid=False)
fig.update_yaxes(showgrid=False)
fig.show()

Number of African countries in the top 15: 4
           Country  Population Added      Continent
228        Vietnam          56258007           Asia
94            Iran          60100865           Asia
57           Egypt          76208117         Africa
131         Mexico          77214819  North America
163    Philippines          78123423           Asia
55        DR Congo          78858479         Africa
63        Ethiopia          95071678         Africa
16      Bangladesh         103644512           Asia
27          Brazil         118943623  South America
221  United States         137961517  North America
93       Indonesia         160272945           Asia
149        Nigeria         162971948         Africa
156       Pakistan         176533990           Asia
41           China         603352887           Asia
92           India         859671872           Asia


---
## Q3 — Do bigger countries have more people?

Is there a relationship between a country's **land area** and its **2022 population** — does it differ by **continent**, and which countries clearly break the pattern?

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

A scatter plot is suitable because it shows the relationship between land area and population across countries. Using continent-based colours and logarithmic scales makes it easier to identify patterns and unusual countries.

In [15]:
# Q3 - Land area and population relationship

labels = ['India', 'China', 'Russia', 'Canada']

df['Label'] = df['Country'].where(
    df['Country'].isin(labels), ''
)

fig = px.scatter(
    df,
    x='Area',
    y='2022 Population',
    color='Continent',
    text='Label',
    log_x=True,
    log_y=True,
    hover_name='Country',
    title='Land area and population are positively related, but not proportionally',
    template='plotly_white',
    color_discrete_sequence=px.colors.qualitative.Safe
)

fig.update_traces(textposition='top center')
fig.update_layout(
    xaxis_title='Land area (km²)',
    yaxis_title='2022 population',
    legend_title=None
)
fig.update_xaxes(showgrid=False)
fig.update_yaxes(showgrid=False)
fig.show()

---
## Q4 — How crowded are the continents?

How does **population density vary within and across continents**? Show the spread, not just the average.

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

A box plot is appropriate because it displays the median, spread and outliers of population density within each continent. This allows us to compare density distributions rather than just average values.

In [16]:
# Q4 - Population density distribution

fig = px.box(
    df,
    x='Continent',
    y='Density',
    color='Continent',
    points='outliers',
    log_y=True,
    color_discrete_sequence=px.colors.qualitative.Safe,
    title='Population density varies greatly within and across continents',
    template='plotly_white'
)

fig.update_layout(
    xaxis_title=None,
    yaxis_title='Population density (people per km²)',
    showlegend=False
)

fig.update_xaxes(showgrid=False)
fig.update_yaxes(showgrid=False)
fig.show()

---
## Q5 — Where did populations grow — and where did they shrink?

Which countries' populations **grew and which shrank between 2000 and 2022** (percentage change), and where in the world are they?

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

A choropleth map is suitable because it displays percentage population changes geographically. A diverging colour scale helps distinguish countries experiencing population growth from those experiencing population decline.

In [17]:
# Q5 - Population percentage change from 2000 to 2022

df['Population Change (%)'] = (
    (df['2022 Population'] - df['2000 Population'])
    / df['2000 Population'] * 100
)

print("Countries that grew:", (df['Population Change (%)'] > 0).sum())
print("Countries that shrank:", (df['Population Change (%)'] < 0).sum())

fig = px.choropleth(
    df,
    locations='CCA3',
    locationmode='ISO-3',
    color='Population Change (%)',
    hover_name='Country',
    color_continuous_scale='PuOr',
    color_continuous_midpoint=0,
    title='Population growth is widespread, while declines cluster in selected regions',
    template='plotly_white'
)

fig.update_layout(
    coloraxis_colorbar_title='Change (%)',
    margin=dict(l=0, r=0, t=60, b=0)
)

fig.show()

Countries that grew: 200
Countries that shrank: 34


---
## Q6 — Where do the world's 8 billion people live?

How is the **2022 world population** distributed across **continents and the countries within them**? Show both levels in one figure.

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

A treemap is appropriate because it displays hierarchical information about continents and their countries in one visualization. The size of each rectangle represents population, making it easy to compare population shares

In [18]:
# Q6 - Population distribution by continent and country

fig = px.treemap(
    df,
    path=['Continent', 'Country'],
    values='2022 Population',
    color='Continent',
    color_discrete_sequence=px.colors.qualitative.Safe,
    title="Asia contains the largest share of the world's population",
    template='plotly_white'
)

fig.update_traces(
    textinfo='label+percent parent',
    hovertemplate=(
        '<b>%{label}</b><br>'
        'Population: %{value:,}<br>'
        '<extra></extra>'
    )
)

fig.update_layout(margin=dict(l=10, r=10, t=60, b=10))
fig.show()

---
## Q7 — Whose share of the world is rising?

How have the shares of world population held by **Africa, Asia and Europe** changed from 1970 to 2022? Group the other three continents as `'Rest of the world'`.

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

A 100% stacked bar chart is suitable because it compares how Africa, Asia, Europe and the rest of the world contribute to the total world population over time. It clearly shows changes in population shares between 1970 and 2022.

In [19]:
continent_year = df_long.groupby(
    ['Continent', 'Year'], as_index=False
)['Population'].sum()

continent_year['Group'] = continent_year['Continent'].replace({
    'North America': 'Rest of the world',
    'South America': 'Rest of the world',
    'Oceania': 'Rest of the world'
})

grouped = continent_year.groupby(
    ['Group', 'Year'], as_index=False
)['Population'].sum()

grouped['World Share'] = (
    grouped['Population'] /
    grouped.groupby('Year')['Population'].transform('sum') * 100
)

fig = px.bar(
    grouped,
    x='Year',
    y='World Share',
    color='Group',
    barmode='stack',
    color_discrete_sequence=px.colors.qualitative.Safe,
    title='Africa’s share of world population is rising',
    template='plotly_white'
)

fig.update_layout(
    xaxis_title=None,
    yaxis_title='Share of world population (%)',
    legend_title=None
)

fig.update_xaxes(showgrid=False)
fig.update_yaxes(showgrid=False, range=[0, 100])
fig.show()